# TRANCE Research Pipeline: Multimodal Readmission Prediction
**Version**: 5.8 (GPU Optimized | Skip-Redundant | Auto-Persist)

--- 
### Step 00: Dependency Bootstrapper
Run this cell first. We install all research-grade dependencies.

In [ ]:
# %% [cell] Step 00: Dependences
!pip install -q transformers torch tqdm scikit-learn pandas numpy matplotlib joblib sentence-transformers sentencepiece

In [ ]:
# %% [cell] Step 0: Setup, Environment & GPU Diagnostics
import os, sys, gc, json, random, re, logging, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from google.colab import drive
from tqdm.auto import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss, precision_recall_curve, roc_curve
from sklearn.calibration import IsotonicRegression
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# 1. Config
RANDOM_SEED = 42
PROJECT_NAME = "TRANCE-Research"
LOCAL_DATA  = "/content/raw_mimic"
LOCAL_MODEL = "/content/clinical_t5" 

# --- UPDATE THESE TO YOUR DRIVE PATHS ---
DRIVE_MIMIC = "/content/drive/MyDrive/WeCare-Model/readmission-ai/data/mimiciv" 
DRIVE_MODEL = "/content/drive/MyDrive/WeCare-Model/readmission-ai/physionet.org/files/clinical-t5/1.0.0/Clinical-T5-Base" 

RESULTS_DIR = "/content/results"
MODELS_DIR  = "/content/models"
EMB_DIM = 768

for d in [LOCAL_DATA, LOCAL_MODEL, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(s=RANDOM_SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

if not os.path.exists('/content/drive'): drive.mount('/content/drive')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'\u2713 Device: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')
else:
    print('\u26a0 NO GPU DETECTED - Fine-Tuning will be slow.')

In [ ]:
# %% [cell] Step 1: High-Speed Ingestion (Data & Model Mirror)
tables = [
    'hosp/admissions.csv.gz', 'hosp/patients.csv.gz', 'hosp/diagnoses_icd.csv.gz', 
    'hosp/labevents.csv.gz', 'note/discharge.csv.gz', 'icu/icustays.csv.gz'
]

print("Mirroring MIMIC-IV subsets to local high-speed SSD...")
for t in tables:
    src = os.path.join(DRIVE_MIMIC, t)
    dest = os.path.join(LOCAL_DATA, os.path.basename(t))
    if os.path.exists(src) and not os.path.exists(dest):
        print(f"  Staging {t}...")
        !cp "{src}" "{dest}"

print("\nMirroring PhysioNet ClinicalT5 to Local SSD...")
if os.path.exists(DRIVE_MODEL) and not os.path.exists(os.path.join(LOCAL_MODEL, 'config.json')):
    print(f"  Mirroring folders to {LOCAL_MODEL}...")
    !cp -rv "{DRIVE_MODEL}"/* "{LOCAL_MODEL}/"
    print("\u2713 Model staged locally.")
elif os.path.exists(os.path.join(LOCAL_MODEL, 'config.json')):
    print("\u2713 Model already exists on local SSD. Skipping mirror.")
else:
    print("\u26a0 PhysioNet folder not found. Attempting Hub (may require HF_TOKEN).")

print("\n\u2714 Ingestion complete.")

In [ ]:
# %% [cell] Step 2: Extraction (With Persistence Check)
KEY_LABS = {
    50912: 'creatinine', 50882: 'bicarb', 50931: 'glucose', 
    50983: 'sodium', 51006: 'bun', 51221: 'hematocrit', 
    50813: 'lactate', 51301: 'wbc'
}

def run_extraction():
    logger.info("Building cohort (Subject-First filtering)...")
    adm = pd.read_csv(os.path.join(LOCAL_DATA, 'admissions.csv.gz'), 
                      usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type'],
                      parse_dates=['admittime','dischtime','deathtime'])
    
    adm = adm.sort_values(['subject_id', 'admittime'])
    adm['next_admittime'] = adm.groupby('subject_id')['admittime'].shift(-1)
    adm['days_to_next'] = (adm['next_admittime'] - adm['dischtime']).dt.total_seconds() / 86400
    adm['readmit_30'] = ((adm['days_to_next'] >= 0) & (adm['days_to_next'] <= 30) & (adm['deathtime'].isna())).astype(int)

    logger.info("Calculating Temporal Trajectories (Slopes/Deltas)...")
    lab_reader = pd.read_csv(os.path.join(LOCAL_DATA, 'labevents.csv.gz'), 
                             usecols=['hadm_id','itemid','valuenum','charttime'],
                             chunksize=5_000_000, parse_dates=['charttime'])
    target_ids = set(adm['hadm_id'])
    labs = pd.concat([c[c['hadm_id'].isin(target_ids) & c['itemid'].isin(KEY_LABS)] for c in tqdm(lab_reader)], ignore_index=True)
    
    trajectories = []
    for hid, g in tqdm(labs.groupby('hadm_id'), desc="Engineering Trajectories"):
        row = {'hadm_id': hid}
        for itid, name in KEY_LABS.items():
            sg = g[g['itemid']==itid].sort_values('charttime')
            if len(sg) > 0:
                row[f'lab_{name}_mean'] = sg['valuenum'].mean()
                row[f'lab_{name}_last'] = sg['valuenum'].values[-1]
                if len(sg) > 1:
                    delta = sg['valuenum'].values[-1] - sg['valuenum'].values[0]
                    dur = (sg.iloc[-1]['charttime']-sg.iloc[0]['charttime']).total_seconds()/86400
                    row[f'lab_{name}_slope'] = delta / max(dur, 0.1)
            else: row[f'lab_{name}_mean'] = 0; row[f'lab_{name}_slope'] = 0
        trajectories.append(row)
    
    df_final = adm.merge(pd.DataFrame(trajectories), on='hadm_id', how='left').fillna(0)
    path_pat = os.path.join(LOCAL_DATA, 'patients.csv.gz')
    if os.path.exists(path_pat):
        pat = pd.read_csv(path_pat, usecols=['subject_id','gender','anchor_age'])
        df_final = df_final.merge(pat, on='subject_id').fillna(0)
    return df_final.drop(columns=['admittime','dischtime','deathtime','next_admittime','days_to_next'])

# PERSISTENCE CHECK: Skip if already extracted
EHR_CACHE = os.path.join(RESULTS_DIR, 'df_master.csv')
if os.path.exists(EHR_CACHE):
    print(f"\u2713 Found existing EHR features at {EHR_CACHE}. Skipping ~1hr extraction.")
    df_master = pd.read_csv(EHR_CACHE)
else:
    df_master = run_extraction()
    df_master.to_csv(EHR_CACHE, index=False)
    print(f"\u2713 Saved features to {EHR_CACHE}.")

print(f"\u2714 Enhanced Feature Space Ready: {df_master.shape[1]} features | {len(df_master)} admissions")

In [ ]:
# %% [cell] Step 3: Multimodal Signal Generation (OFFLINE PATH)
class NoteDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts = [str(t) for t in texts]; self.tok = tokenizer; self.ml = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        e = self.tok(self.texts[idx], max_length=self.ml, padding='max_length', truncation=True, return_tensors='pt')
        return e['input_ids'].squeeze(0), e['attention_mask'].squeeze(0)

def setup_multimodal(df_master):
    logger.info("Linking Notes to Cohort...")
    note_path = os.path.join(LOCAL_DATA, 'discharge.csv.gz')
    if not os.path.exists(note_path): return df_master
    notes = pd.read_csv(note_path, usecols=['hadm_id','text'])
    df = df_master.merge(notes, on='hadm_id', how='inner')
    
    if os.path.exists(os.path.join(LOCAL_MODEL, 'config.json')):
        MODEL_ID = LOCAL_MODEL
        local_only = True
    else:
        MODEL_ID = "GanjinZero/clinical-t5-base"
        local_only = False
    
    tok  = T5Tokenizer.from_pretrained(MODEL_ID, local_files_only=local_only, use_fast=False, legacy=False)
    enc  = T5ForConditionalGeneration.from_pretrained(MODEL_ID, local_files_only=local_only).to(DEVICE)
    
    logger.info("Generating 768-dim baseline embeddings...")
    ds = NoteDataset(df['text'].values, tok)
    dl = DataLoader(ds, batch_size=16, shuffle=False)
    
    emb_list = []
    enc.eval()
    with torch.no_grad(), autocast(enabled=True):
        for ids, msk in tqdm(dl):
            out = enc.encoder(input_ids=ids.to(DEVICE), attention_mask=msk.to(DEVICE)).last_hidden_state
            emb_list.append(out.mean(dim=1).cpu().numpy())
            
    df_emb = pd.DataFrame(np.vstack(emb_list), columns=[f'ct5_{i}' for i in range(EMB_DIM)])
    df_final = pd.concat([df.reset_index(drop=True), df_emb.reset_index(drop=True)], axis=1)
    return df_final.drop(columns=['text'])

# MULTIMODAL CACHE
MULTIMODAL_CACHE = os.path.join(RESULTS_DIR, 'df_multimodal.csv')
if os.path.exists(MULTIMODAL_CACHE):
    print(f"\u2713 Found existing multimodal data at {MULTIMODAL_CACHE}. Skipping embedding generation.")
    df_multimodal = pd.read_csv(MULTIMODAL_CACHE)
else:
    df_multimodal = setup_multimodal(df_master)
    df_multimodal.to_csv(MULTIMODAL_CACHE, index=False)
    print(f"\u2713 Saved multimodal dataset to {MULTIMODAL_CACHE}.")

print(f"\u2714 Multimodal Dataset Size: {df_multimodal.shape[1]} features | {len(df_multimodal)} admissions")

In [ ]:
# %% [cell] Step 4: TRANCE-Gate Model Architecture
class TextGuidedGate(nn.Module):
    def __init__(self, text_dim=768, tabular_dim=30):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(text_dim, 128), nn.ReLU(), 
                                nn.Linear(128, tabular_dim), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(text_dim + tabular_dim, 256), nn.ReLU(), 
                                       nn.Dropout(0.3), nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, text_emb, x_tab, return_gate=False):
        w = self.gate(text_emb)
        f = torch.cat([text_emb, w * x_tab], dim=1)
        out = self.classifier(f).squeeze(1)
        return (out, w) if return_gate else out

print(f"\n{'='*45}")
print(f"TRANCE-GATE SYSTEM ACTIVE (Physicsnet Offline Enabled)")
print(f"{'='*45}")

In [ ]:
# %% [cell] Step 5: High-Performance Training Loop (Gated Fusion)
def run_training_gated(df):
    logger.info("Preparing Research-Grade Training Loop...")
    # Standardize data types
    tab_cols = [c for c in df.columns if 'lab_' in c or 'anchor_' in c or 'gender' in c]
    txt_cols = [f'ct5_{i}' for i in range(EMB_DIM)]
    
    if 'gender' in df.columns and df['gender'].dtype == object: 
        df['gender'] = (df['gender'] == 'M').astype(int)
        
    X_tab = df[tab_cols].values.astype(np.float32)
    X_txt = df[txt_cols].values.astype(np.float32)
    y = df['readmit_30'].values.astype(np.float32)
    
    logger.info(f"Input Dimensions: Tabular={X_tab.shape[1]} | Text={X_txt.shape[1]}")
    
    gkf = GroupKFold(n_splits=5)
    train_idx, val_idx = next(gkf.split(X_tab, y, df['subject_id']))
    
    def to_t(arr): return torch.tensor(arr).to(DEVICE)
    
    model = TextGuidedGate(text_dim=EMB_DIM, tabular_dim=X_tab.shape[1]).to(DEVICE)
    opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scaler = GradScaler()
    criterion = nn.BCELoss()
    
    best_auc = 0
    
    for epoch in range(1, 11):
        model.train()
        indices = np.random.permutation(train_idx)
        batch_size = 64
        losses = []
        
        for i in range(0, len(indices), batch_size):
            bi = indices[i:i+batch_size]
            opt.zero_grad()
            with autocast(enabled=True):
                pred = model(to_t(X_txt[bi]), to_t(X_tab[bi]))
                loss = criterion(pred, to_t(y[bi]))
            
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
            
        model.eval()
        with torch.no_grad():
            v_pred = model(to_t(X_txt[val_idx]), to_t(X_tab[val_idx])).cpu().numpy()
            v_auc  = roc_auc_score(y[val_idx], v_pred)
            
        print(f"Epoch {epoch:02d} | Loss: {np.mean(losses):.4f} | Val AUROC: {v_auc:.4f}")
        if v_auc > best_auc:
            best_auc = v_auc
            torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'trance_gate_best.pt'))
            
    print(f"\n{'='*45}")
    print(f"BEST AUROC ACHIEVED: {best_auc:.4f}")
    print(f"{'='*45}")
    return model

final_model = run_training_gated(df_multimodal)

In [ ]:
# %% [cell] Step 6: Benchmark Analysis (Calibration & Fairness)
def run_benchmark(model, df):
    logger.info("Generating Research Benchmarks...")
    tab_cols = [c for c in df.columns if 'lab_' in c or 'anchor_' in c or 'gender' in c]
    txt_cols = [f'ct5_{i}' for i in range(EMB_DIM)]
    
    X_tab = torch.tensor(df[tab_cols].values.astype(np.float32)).to(DEVICE)
    X_txt = torch.tensor(df[txt_cols].values.astype(np.float32)).to(DEVICE)
    y_true = df['readmit_30'].values
    
    model.eval()
    with torch.no_grad():
        y_prob, weights = model(X_txt, X_tab, return_gate=True)
        y_prob = y_prob.cpu().numpy()
        weights = weights.cpu().numpy()
        
    auc = roc_auc_score(y_true, y_prob)
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'AUC={auc:.3f}')
    plt.plot([0,1],[0,1],'k--')
    plt.title("ROC Curve")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.bar(range(X_tab.shape[1]), weights.mean(axis=0))
    plt.xticks(range(X_tab.shape[1]), tab_cols, rotation=90)
    plt.title("Gated Tabular Weights")
    plt.tight_layout()
    plt.show()

run_benchmark(final_model, df_multimodal)